In [12]:
# ================================================
# 03 - PLAYER RANKING
# Scoring portal players for NC State fit
# ================================================

import pandas as pd

# Load our cleaned file
df = pd.read_csv("data/cleaned/portal_cleaned.csv")

print(f"Players loaded: {len(df)}")
print(f"Players with BPR: {df['BPR'].notna().sum()}")
print(f"Players with DBPR: {df['DBPR'].notna().sum()}")
print("\nReady to build ranking model!")

# -- SCORING WEIGHTS --
# Defense heavy to mirror Gainey's system priority
# Driscoll offense still matters but defense comes first
# Adjust these anytime to change ranking emphasis

WEIGHT_DBPR        = 0.40  # Defensive rating (Gainey priority)
WEIGHT_OBPR        = 0.25  # Offensive rating (Driscoll system)
WEIGHT_BPR         = 0.20  # Overall player quality
WEIGHT_COMPETITION = 0.15  # Strength of schedule adjustment

print("Scoring weights:")
print(f"  Defensive DBPR:     {WEIGHT_DBPR*100:.0f}%")
print(f"  Offensive OBPR:     {WEIGHT_OBPR*100:.0f}%")
print(f"  Overall BPR:        {WEIGHT_BPR*100:.0f}%")
print(f"  Competition level:  {WEIGHT_COMPETITION*100:.0f}%")
print(f"  Total:              {(WEIGHT_DBPR+WEIGHT_OBPR+WEIGHT_BPR+WEIGHT_COMPETITION)*100:.0f}%")

# -- BUILD THE SCORING FUNCTION --
# We normalize each metric to a 0-100 scale
# so that DBPR and OBPR can be fairly compared
# Then apply our weights to get a final composite score

from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Only score players who have advanced metrics
rated = df[df['BPR'].notna()].copy()
unrated = df[df['BPR'].isna()].copy()

print(f"Players we can score: {len(rated)}")
print(f"Players without metrics: {len(unrated)}")

# Normalize each metric to 0-100 scale
scaler = MinMaxScaler(feature_range=(0, 100))

rated['DBPR_score']   = scaler.fit_transform(rated[['DBPR']])
rated['OBPR_score']   = scaler.fit_transform(rated[['OBPR']])
rated['BPR_score']    = scaler.fit_transform(rated[['BPR']])
rated['COMP_score']   = scaler.fit_transform(rated[['Adj_Team_Eff_Margin']])

# Apply weights to get composite score
rated['composite_score'] = (
    rated['DBPR_score']  * WEIGHT_DBPR +
    rated['OBPR_score']  * WEIGHT_OBPR +
    rated['BPR_score']   * WEIGHT_BPR  +
    rated['COMP_score']  * WEIGHT_COMPETITION
)

print("\nScoring complete!")
print(f"Score range: {rated['composite_score'].min():.1f} - {rated['composite_score'].max():.1f}")


Players loaded: 290
Players with BPR: 149
Players with DBPR: 149

Ready to build ranking model!
Scoring weights:
  Defensive DBPR:     40%
  Offensive OBPR:     25%
  Overall BPR:        20%
  Competition level:  15%
  Total:              100%
Players we can score: 149
Players without metrics: 141

Scoring complete!
Score range: 7.5 - 90.9


In [13]:
# -- TOP 25 RANKED PLAYERS --
# Sort by composite score, show the best available

top25 = rated.sort_values('composite_score', ascending=False).head(25)

print("=== TOP 25 PORTAL PLAYERS (NC State Fit) ===")
print(f"{'Rank':<5} {'Name':<22} {'Pos':<5} {'Team':<20} {'Score':<8} {'DBPR':<7} {'OBPR':<7} {'BPR'}")
print("-" * 80)

for i, (_, row) in enumerate(top25.iterrows(), 1):
    print(f"{i:<5} {row['Name']:<22} {row['Position']:<5} {str(row['Team']):<20} {row['composite_score']:<8.1f} {row['DBPR']:<7.2f} {row['OBPR']:<7.2f} {row['BPR']:.2f}")


=== TOP 25 PORTAL PLAYERS (NC State Fit) ===
Rank  Name                   Pos   Team                 Score    DBPR    OBPR    BPR
--------------------------------------------------------------------------------
1     Milan Momcilovic       PF    Iowa State           90.9     2.45    6.06    8.50
2     Baba Miller            PF    Cincinnati           87.0     3.95    3.28    7.23
3     RJ Godfrey             SF    Clemson              83.2     3.16    3.44    6.59
4     Allen Graves           PF    Santa Clara          81.9     2.07    5.58    7.64
5     Ernest Udeh Jr.        C     Miami (Fla.)         77.4     3.24    1.95    5.20
6     Skyy Clark             PG    UCLA                 75.3     1.79    4.23    6.02
7     Malique Ewin           PF    Arkansas             74.3     1.39    4.38    5.77
8     Tyler Nickel           SF    Vanderbilt           73.9     1.02    4.99    6.02
9     Jalen Washington       PF    Vanderbilt           73.5     2.22    2.83    5.05
10    Joseph Pi

In [14]:
# -- STRATEGY FOR UNRATED PLAYERS --
# Group them so we know how to handle each case

print("=== UNRATED PLAYERS BY POSITION ===")
print(unrated['Position'].value_counts())

print("\n=== UNRATED PLAYERS WITH ON3 RATING ===")
has_on3 = unrated[unrated['ON3 Rating'].notna()]
print(f"Have ON3 rating: {len(has_on3)}")
print(f"No ON3 rating either: {len(unrated) - len(has_on3)}")

print("\n=== UNRATED PLAYERS WITH ON3 - SAMPLE ===")
print(has_on3[['Name', 'Position', 'Year', 'ON3 Rating']].sort_values(
    'ON3 Rating', ascending=False).head(10))


=== UNRATED PLAYERS BY POSITION ===
SG    52
PF    36
PG    19
SF    17
C     16
Name: Position, dtype: int64

=== UNRATED PLAYERS WITH ON3 RATING ===
Have ON3 rating: 25
No ON3 rating either: 116

=== UNRATED PLAYERS WITH ON3 - SAMPLE ===
                Name Position   Year  ON3 Rating
10     Quadir Martin       PF     SO        92.0
12    Daeshun Ruffin       PG  RS-SR        91.0
11  Mihailo Petrovic       PG     FR        91.0
15       Ryan Blount       SF     SO        90.0
18  Ja'Quay Randolph        C     JR        89.0
19   Lazerek Houston       PG     FR        89.0
21  Bernie Blunt III       PG     SR        89.0
23      Daquan Davis       PG     SO        89.0
26  Anthony Robinson       PF     JR        89.0
30      JoJo Wallace       SG     SO        89.0


In [15]:
# -- TOP 25 PER POSITION --

positions = ['PG', 'SG', 'SF', 'PF', 'C']

for pos in positions:
    pos_players = rated[rated['Position'] == pos].sort_values(
        'composite_score', ascending=False).head(25)
    
    print(f"\n=== TOP 25 {pos} IN PORTAL ===")
    print(f"{'Rank':<5} {'Name':<22} {'Team':<22} {'Score':<8} {'DBPR':<7} {'OBPR':<7} {'BPR'}")
    print("-" * 75)
    
    for i, (_, row) in enumerate(pos_players.iterrows(), 1):
        name = str(row['Name'])
        team = str(row['Team'])
        score = row['composite_score']
        dbpr = row['DBPR']
        obpr = row['OBPR']
        bpr = row['BPR']
        print(f"{i:<5} {name:<22} {team:<22} {score:<8.1f} {dbpr:<7.2f} {obpr:<7.2f} {bpr:.2f}")
    
    total = len(rated[rated['Position'] == pos])
    print(f"Total {pos} available: {total}")



=== TOP 25 PG IN PORTAL ===
Rank  Name                   Team                   Score    DBPR    OBPR    BPR
---------------------------------------------------------------------------
1     Skyy Clark             UCLA                   75.3     1.79    4.23    6.02
2     Jayden Pierre          TCU                    70.7     2.16    2.54    4.70
3     Trey Campbell          Northern Iowa          70.1     2.56    2.11    4.68
4     Jordan Pope            Texas                  64.9     0.56    4.11    4.67
5     Seth Trimble           North Carolina         63.1     0.99    3.06    4.04
6     Kenyon Giles           Wichita State          62.3     1.00    3.10    4.10
7     Derek Simpson          Saint Joseph's         61.6     1.42    2.68    4.10
8     Tre Holloman           NC State               60.2     0.95    2.46    3.40
9     Desmond Claude         Washington             58.2     0.87    2.15    3.02
10    Isaiah Watts           Maryland               56.7     1.63    0.79   

In [16]:
# -- TIER 2: UNRATED PLAYERS WITH ON3 RATING --
# These players don't have advanced metrics
# but we can use their ON3 rating as a proxy score
# ON3 ratings are on a 0-100 scale already

# Get unrated players who have an ON3 rating
tier2 = unrated[unrated['ON3 Rating'].notna()].copy()

print(f"Tier 2 players (have ON3 rating): {len(tier2)}")
print(f"Tier 3 players (no metrics at all): {len(unrated) - len(tier2)}")

# Use ON3 rating directly as their score
# but cap it at 85 so they don't outrank fully rated players
tier2['composite_score'] = tier2['ON3 Rating'].clip(upper=85)
tier2['score_source'] = 'ON3 Only'

print("\n=== TIER 2 TOP 15 ===")
print(f"{'Name':<22} {'Pos':<6} {'Year':<8} {'ON3':<8} {'Score'}")
print("-" * 55)

top_tier2 = tier2.sort_values('composite_score', ascending=False).head(15)
for _, row in top_tier2.iterrows():
    print(f"{row['Name']:<22} {row['Position']:<6} {row['Year']:<8} {row['ON3 Rating']:<8.1f} {row['composite_score']:.1f}")


Tier 2 players (have ON3 rating): 25
Tier 3 players (no metrics at all): 116

=== TIER 2 TOP 15 ===
Name                   Pos    Year     ON3      Score
-------------------------------------------------------
Quadir Martin          PF     SO       92.0     85.0
Tre Norman             SG     JR       88.0     85.0
Omar Adegbola          SG     RS-SO    85.0     85.0
Travonne Jackson       SG     RS-JR    85.0     85.0
Reid Ducharme          SG     JR       86.0     85.0
Jahki Howard           SF     SO       87.0     85.0
Alfred Worrell         SG     SR       87.0     85.0
Will Fowler            SG     SO       87.0     85.0
Antonio Pusateri       PF     JR       88.0     85.0
Mike Mora              SG     RS-SO    88.0     85.0
Jevon Hill             PF     SR       88.0     85.0
Marqus Mitrovic Marion PF     RS-SO    88.0     85.0
Justin Payne           SG     SO       88.0     85.0
Mihailo Petrovic       PG     FR       91.0     85.0
Robert Davis           SG     JR       88.0     

In [17]:
# -- ADJUST TIER 2 CAP --
# 85 cap is too low, compressing everyone together
# Set cap to 92 so ON3 differences show through
# but still keeps them below our best fully rated players

tier2['composite_score'] = tier2['ON3 Rating'].clip(upper=92)
tier2['score_source'] = 'ON3 Only'

print("=== TIER 2 BY POSITION ===")

positions = ['PG', 'SG', 'SF', 'PF', 'C']

for pos in positions:
    pos_players = tier2[tier2['Position'] == pos].sort_values(
        'composite_score', ascending=False)
    
    if len(pos_players) == 0:
        continue
        
    print(f"\n--- {pos} ---")
    for _, row in pos_players.iterrows():
        print(f"  {row['Name']:<25} ON3: {row['ON3 Rating']:.1f}  Score: {row['composite_score']:.1f}  Year: {row['Year']}")


=== TIER 2 BY POSITION ===

--- PG ---
  Mihailo Petrovic          ON3: 91.0  Score: 91.0  Year: FR
  Daeshun Ruffin            ON3: 91.0  Score: 91.0  Year: RS-SR
  Lazerek Houston           ON3: 89.0  Score: 89.0  Year: FR
  Bernie Blunt III          ON3: 89.0  Score: 89.0  Year: SR
  Daquan Davis              ON3: 89.0  Score: 89.0  Year: SO

--- SG ---
  JoJo Wallace              ON3: 89.0  Score: 89.0  Year: SO
  Trey Simmons              ON3: 88.0  Score: 88.0  Year: SO
  Robert Davis              ON3: 88.0  Score: 88.0  Year: JR
  Justin Payne              ON3: 88.0  Score: 88.0  Year: SO
  Tre Norman                ON3: 88.0  Score: 88.0  Year: JR
  Mike Mora                 ON3: 88.0  Score: 88.0  Year: RS-SO
  Will Fowler               ON3: 87.0  Score: 87.0  Year: SO
  Alfred Worrell            ON3: 87.0  Score: 87.0  Year: SR
  Reid Ducharme             ON3: 86.0  Score: 86.0  Year: JR
  Travonne Jackson          ON3: 85.0  Score: 85.0  Year: RS-JR
  Omar Adegbola          

In [18]:
# -- TIER 3 NOTE --
# Development prospects for PF and C
# Players with no metrics but strong physical profile
# To be built in 04_nil_valuation or 05_team_fit
# Focus on: height 6-8+, weight 220+, athleticism indicators
# These are project players for Gainey's development system

tier3 = unrated[
    (unrated['Position'].isin(['PF', 'C'])) &
    (unrated['ON3 Rating'].isna())
].copy()

print(f"Tier 3 development prospects (PF/C unrated): {len(tier3)}")
print("\n=== PHYSICAL PROFILES ===")
print(tier3[['Name', 'Position', 'Year', 'Height', 'Weight']].to_string())


# -- SAVE TIER 3 DEVELOPMENT PROSPECTS --

tier3['tier'] = 'Tier 3 - Development'
tier3['score_source'] = 'Physical Profile Only'
tier3['composite_score'] = None

tier3.to_csv("data/outputs/tier3_development.csv", index=False)

print(f"Tier 3 saved: {len(tier3)} development prospects")
print(f"Centers: {len(tier3[tier3['Position'] == 'C'])}")
print(f"Power Forwards: {len(tier3[tier3['Position'] == 'PF'])}")


Tier 3 development prospects (PF/C unrated): 46

=== PHYSICAL PROFILES ===
                   Name Position   Year Height  Weight
79    Vincent Iwuchukwu        C     SR   6-11   220.0
84       Isaiah Miranda       PF     SR    7-1   205.0
85         Niko Bundalo       PF     FR   6-10   205.0
95          Sam Walters       PF     JR    6-9   180.0
104          Papa Kante        C     JR   6-10   225.0
109      Dylan Anderson       PF  RS-JR    7-0   215.0
113       Jordan Butler       PF     JR    7-0   240.0
114    Emmanuel Stephen        C     SO    7-0   215.0
121       Macaleab Rich       PF     JR    6-7   225.0
127      Jack McCaffery       PF     FR    6-8   200.0
137          Kebba Njie       PF     SR   6-10   237.0
148        Jevon Porter       PF     SR   6-11   230.0
150       Steven Solano        C     SO   6-11   230.0
152      Cameron Barnes       PF     JR   6-10   210.0
153  Godswill Erheriene        C     SO    6-9   225.0
165         Amdy Ndiaye       PF     SO    6-

In [19]:
# -- COMBINE ALL TIERS INTO MASTER RANKINGS --

# Add score source to tier 1
rated['score_source'] = 'Full Metrics'

# Add tier labels
rated['tier'] = 'Tier 1'
tier2['tier'] = 'Tier 2'
tier3['tier'] = 'Tier 3 - Development'
tier3['score_source'] = 'Physical Profile Only'
tier3['composite_score'] = None

# Combine all three tiers
master = pd.concat([rated, tier2, tier3], ignore_index=True)

print(f"Total players in master list: {len(master)}")
print(f"Tier 1 (full metrics):        {len(rated)}")
print(f"Tier 2 (ON3 only):            {len(tier2)}")
print(f"Tier 3 (development):         {len(tier3)}")

# Save master rankings
master.to_csv("data/outputs/master_rankings.csv", index=False)
print("\nMaster rankings saved!")


Total players in master list: 220
Tier 1 (full metrics):        149
Tier 2 (ON3 only):            25
Tier 3 (development):         46

Master rankings saved!
